## Disclaimer

**Environment:** This notebook has been tested with:

- Julia v1.12.x
- JuMP.jl v1.30.x
- HiGHS v1.24.1

**Author:** Xavier Gandibleux, Nantes (France)


----

## Case 2: Multi-Objective Optimization


**Summary of main Notations:**

| Notation | Description |
|---|---|
| $\mathbb{R}^n$ | The decision space, with $n$ the number of variables |
| $\mathbb{R}^p$ | The objective space, with $p$ the number of objectives |
| $X$ | The set of feasible solutions in $\mathbb{R}^n$ |
| $Y = z(X)$ | The set of images of feasible solutions in $\mathbb{R}^p$ |
| $X_E \subseteq X$ | A complete set of efficient solutions |
| $Y_N \subseteq Y$ | The set of nondominated points, with $Y_N = Y_{SN} \cup Y_{NN}$ |
| $Y_{SN} \subseteq Y_N$ | The set of supported nondominated points |
| $Y_{NN} \subseteq Y_N$ | The set of nonsupported nondominated points |
| $y^I \in \mathbb{R}^p$ | The ideal point |
| $y^U \in \mathbb{R}^p$ | The utopian point, with (in $\max$) $y^U = y^I + \epsilon$ |
| $y^N \in \mathbb{R}^p$ | The nadir point |

<br>
<br>

**The base bi-objective model:**

Given the bi-objective 01 Unidimensional Knapsack Problem (bi-01UKP) formulated by 
$$z=\max\big\{(p^1x,\ p^2x) \mid wx \le c, \ x\in\{0,1\}^n\big\}$$

and the numerical instance corresponding to 
$$n=10$$
$$p^1=(13, \ 10, \ 3, \ 16, \ 12, \ 11, \ 1, \ 9, \ 19, \ 13)$$
$$p^2=(1, 10, \ 3, \ 13, \ 12, \ 19, \ 16, \ 13, \ 11, \ 9)$$
$$w=(\ 4, \ 4, \ 3, \ 5, \ 5, \ 3, \ 2, \ 3, \ 5, \ 4)$$
$$c=19$$

perform the following progressive tasks:

1. Using the package `JuMP`, implement an implicit formulation of the bi-01UKP and display the model.  

2. Using `HiGHS` as MIP solver, the package `MultiObjectiveAlgorithms` and the algorithm `MOA.EpsilonConstraint()`, compute and display $X_E$ and $Y_N$, a complete set of efficient solutions and the set of nondominated points.
3. Switch to the algorithm `MOA.Dichotomy()` without rebuilding the model, and re-optimize to get $Y_{SN}$, the supported nondominated points. How many points are found now? Why fewer?
4. Identify $Y_{NN} = Y_N \setminus Y_{SN}$, the nonsupported nondominated points.
5. From $Y_N$, compute the ideal point $y^I$ and the nadir point $y^N$.
6. Perform a manual scalarization in building a single-objective JuMP model (without `MultiObjectiveAlgorithms`) that maximizes the weighted sum $0.5\,p^1x + 0.5\,p^2x$ directly with `HiGHS`. Check whether the resulting point belongs to $Y_{SN}$.
7. Using the package `Plots`, display $Y_N$ in the objective space $\mathbb{R}^p$, in differenting points belonging to $Y_{SN}$ and $Y_{NN}$.
8. Extending the 01 unidimensionnal knapsack problem to the tri-objective case with the given vector coefficients, report $Y_N$, $Y_{SE}$, and $Y_{NN}$. Display $Y_N$ in the objective space $\mathbb{R}^p$.


<br> 

**Summary of in-place model modification commands:**

Reusing the same `model` object avoids unnecessary reconstruction:

| Change needed | Command (no rebuild) |
|---|---|
| Change MOA algorithm | `set_optimizer(model, MOA.Algorithm(), new_algo`) |
| Change one objective function | `@objective(model, Max, [obj1, new_obj2])` |
| Add a constraint | `@constraint(model, name, expr)` |
| Remove a constraint | `delete(model, mycon)` & `unregister(model, :mycon)` |
| Change RHS of a constraint | `set_normalized_rhs(con, new_value)` |
| Change a constraint coefficient | `set_normalized_coefficient(con, var, new_value)` |


## Solutions:

----
### 1. using the package `JuMP`, implement an implicit formulation of the bi-01UKP and display the model.

#### • declare the package(s) to use:

In [ ]:
using JuMP, HiGHS
import MultiObjectiveAlgorithms as MOA

<br>

#### • setup the instance to solve

In [ ]:
p1 = [ 13, 10,  3, 16, 12, 11,  1,  9, 19, 13 ] # profit 1
p2 = [  1, 10,  3, 13, 12, 19, 16, 13, 11,  9 ] # profit 2
w  = [  4,  4,  3,  5,  5,  3,  2,  3,  5,  4 ] # weight
c  = 19                                         # capacity
n  = length(p1)                                 # number of items

<br>

#### • setup the formulation

In [ ]:
mo01UKP = Model( )

@variable( mo01UKP, x[1:n], Bin )

@expression( mo01UKP, objective1, sum( p1[i] * x[i] for i in 1:n ) )
@expression( mo01UKP, objective2, sum( p2[i] * x[i] for i in 1:n ) )
@objective( mo01UKP, Max, [objective1, objective2] )

@constraint( mo01UKP, capacity, sum( w[i] * x[i] for i in 1:n ) ≤ c )

print( mo01UKP )

<br>

---- 
### 2. using the package `MultiObjectiveAlgorithms`, compute and display $X_E$ and $Y_N$, a complete set of efficient solutions and the set of nondominated points.

#### • setup the MIP solver to use
<ins>Hints:</ins> see `set_optimizer()` in the slides

In [ ]:
set_optimizer(mo01UKP, () -> MOA.Optimizer(HiGHS.Optimizer))

<br>

#### • setup the algorithm to use
<ins>Hints:</ins> see `set_attribute()` in the slides

In [ ]:
set_attribute(mo01UKP, MOA.Algorithm(), MOA.EpsilonConstraint())

<br>

#### • solve the problem

In [ ]:
set_silent(mo01UKP)
optimize!(mo01UKP)
@assert is_solved_and_feasible(mo01UKP)

<br>

#### • query the number of points in $Y_N$
<ins>Hints:</ins> see `result_count()` in the slides

In [ ]:
println("|Y_N| = ", result_count(mo01UKP) )

<br>

#### • query the vector values of the 3rd point

In [ ]:
objective_value(mo01UKP; result = 3)

<br>

#### • query the value for the first objective of the 3rd point

In [ ]:
value(objective1; result = 3)

<br>

#### • display $X_E$ and $Y_N$
For each result, print the efficient solutions ($x \in X_E$) and its image in objective space ($y \in Y_N$).

In [ ]:
for i in 1:result_count(mo01UKP)
    xe = round.(Int, value.(x; result=i))
    yn = round.(Int, objective_value(mo01UKP; result = i))
    println(i, ": x = ", xe, " | z = ", yn)
end

<br>

---- 
### 3. switch algorithm without rebuilding the model

Reuse the same `mo01UKP` object, switch the algorithm to `MOA.Dichotomy()`, and re-optimize to get $Y_{SN}$, the supported nondominated points.  How many points are found now? Why fewer?
Are we certain to obtain all the supported nondominated points with `MOA.Dichotomy()`?

In [ ]:
set_attribute(mo01UKP, MOA.Algorithm(), MOA.Dichotomy())
optimize!(mo01UKP)

card_Y_SN = result_count(mo01UKP)
Y_SN = [round.(Int, objective_value(mo01UKP; result = i)) for i in 1:card_Y_SN]
println("|Y_SN| = ", card_Y_SN)
println("Y_SN   = ", Y_SN)

**Answer 1:** `Dichotomy` only finds points in $Y_{SN}$ (those obtainable by weighted-sum scalarization), so $|Y_{SN}| \le |Y_N|$.

**Answer 2:** No, the non-dominated supported points are divided into extreme points $Y_{SN_1}$ (located at the edges of the polyhedron) and non-extreme points $Y_{SN_2}$. This algorithm only guarantees that all extreme supported points will be found. In general, there are also non-extreme supported points, some of whom can be found using this algorithm, but there is no guarantee that all of them will be found. Activating another algorithm (for enumerating non-extreme supported solutions) will be necessary to ensure the completeness of $Y_{SN}$.

A simple principle for the 0-1 knapsack: nonextreme supported points can be enumerated using a K-best (ranking) algorithm: run the standard dynamic-programming recursion on the scalarized profit for each pair of consecutive extreme supported points, then backtrack through all — rather than just one — optimal paths in the DP table achieving the same optimal value.


<br>

---- 
### 4. identify $Y_{NN} = Y_N \setminus Y_{SN}$, the nonsupported nondominated points.

Switch back to `MOA.EpsilonConstraint()`, recompute $Y_N$, and compute $Y_{NN} = Y_N \setminus Y_{SN}$.  

<ins>Hints:</ins> see `setdiff()` in the Julia documentation


In [ ]:
set_attribute(mo01UKP, MOA.Algorithm(), MOA.EpsilonConstraint())
optimize!(mo01UKP)

card_Y_N = result_count(mo01UKP)
Y_N = [round.(Int, objective_value(mo01UKP; result = i)) for i in 1:card_Y_N]

Y_NN = setdiff(Y_N, Y_SN)
card_Y_NN = length(Y_NN)
println("|Y_NN| = ", card_Y_NN)
println("Y_NN   = ", Y_NN)

**Comment:** The approach used here to identify $Y_{SN}$ and $Y_{NN}$ from $Y_N$ is illustrative and is not generally recommended. Indeed, given $Y_N$, calculating $Y_{SN}$ from scratch to identify the two sets leads to redundant computations. The appropriate approach would be to extract $Y_{SN}$ directly from $Y_N$ as follow.

A simple principle: Given $Y_N$ sorted by objective 1 ascending (objective 2 descending), the monotone-chain sweep tests, for each consecutive triplet $(a,b,c)$, whether 
$b$ lies strictly inside the convex chain formed by $a$ and $c$ (cross-product test):

- If the test is strictly positive (concave turn) → $b$ is dominated by the combination of $a$ and $c$: it is not a supported point, so it must be removed.
- If the test is zero (the 3 points are collinear) → $b$ lies on the same segment as $a$ and $c$: it is a nonextreme supported point (coptimal with 
$a$ and $c$ for the same weight $w$), so it must be kept.
- If the test is strictly negative (convex turn) → $b$ is an extreme supported point, to be kept.

Algorithm:
1. Sort $Y_N$ by objective 1 ascending.
2. Sweep through the points maintaining a stack: for each new point, pop the previous point while the last turn is not convex (cross-product test).
3. Remaining points in the stack form $Y_{SN}$.

In [ ]:
sorted_Y = sort(Y_N, by = y -> (y[1], -y[2]))

Y_SN = [sorted_Y[1]]
for y in sorted_Y[2:end]
    while length(Y_SN) >= 2 &&
          # cross(a,b,c) = (b1​−a1​)(c2​−a2​) − (b2​−a2​)(c1​−a1​)
          (Y_SN[end][1]-Y_SN[end-1][1])*(y[2]-Y_SN[end-1][2]) - (Y_SN[end][2]-Y_SN[end-1][2])*(y[1]-Y_SN[end-1][1]) > 0 
        pop!(Y_SN)
    end
    push!(Y_SN, y)
end

Y_NN = setdiff(Y_N, Y_SN)

println("|Y_N|  = ", length(Y_N))
println("|Y_SN| = ", length(Y_SN))
println("|Y_NN| = ", length(Y_NN))

<br>

---- 
### 5. From $Y_N$, compute the ideal point $y^I$ and the nadir point $y^N$.
The ideal point $y^I$ is the best value per objective. And the nadir point $y^N$ is the worst value among nondominated points.

<ins>Hints:</ins> see `maximum()` and `minimum()` in the Julia documentation

In [ ]:
z1_vals = [y[1] for y in Y_N]
z2_vals = [y[2] for y in Y_N]

zI = [maximum(z1_vals), maximum(z2_vals)]
zN = [minimum(z1_vals), minimum(z2_vals)]

println("Ideal point: ", zI)
println("Nadir point: ", zN)

**Comment:** In the bi-objective case, the nadir point can be obtained immediately from the lexicographically optimal solutions. With more than two objectives, finding the nadir point is generally difficult (see: Ehrgott, M., Tenfelde-Podehl, D., 2003. Computation of ideal and nadir values and implications for their use in MCDM methods. *European Journal of Operational Research* 151, 119–139).

<br>

---- 
### 6. Manual scalarization

Perform a manual scalarization in building a single-objective JuMP model (without `MultiObjectiveAlgorithms`) that maximizes the weighted sum $0.5\,p^1x + 0.5\,p^2x$ directly with HiGHS. Check whether the resulting point belongs to $Y_{SN}$.

In [ ]:
using JuMP, HiGHS

model_scalar = Model(HiGHS.Optimizer)
set_silent(model_scalar)

@variable(model_scalar, xs[1:n], Bin)
@objective(model_scalar, Max, sum(0.5*p1[i]*xs[i] + 0.5*p2[i]*xs[i] for i in 1:n))
@constraint(model_scalar, sum(w[i]*xs[i] for i in 1:n) <= c)

optimize!(model_scalar)

x_sol = round.(Int, value.(xs))
y_sol = [sum(p1 .* x_sol), sum(p2 .* x_sol)]

println("Weighted-sum solution x ∈ X : ", x_sol)
println("Corresponding y = (p1x, p2x) : ", y_sol)
println("y ∈ Y_SN ? ", y_sol in Y_SN)

**Note:** changing the weights (e.g. `0.2`/`0.8`, `0.8`/`0.2`) and re-running this model shows how different weight vectors recover different points of $Y_{SN}$ — but **never** a point of $Y_{NN}$, illustrating why weighted-sum scalarization alone cannot generate the full nondominated set.

<br>

----
### 7. using the package `Plots`, display $Y_N$ in the objective space $Y$
Using `Plots.jl`, plot $Y_N$, marking $Y_{SN}$ and $Y_{NN}$ with different colors, $y^I$, and $y^N$ with different markers/colors

#### • declare the package to use:

In [ ]:
using Plots

#### plot $Y_{SN}$, $Y_{NN}$, $y^I$, and $y^N$ in $\mathbb{R}^p$:

In [ ]:
z1_SN = [y[1] for y in Y_SN] ; z2_SN = [y[2] for y in Y_SN]
z1_NN = [y[1] for y in Y_NN] ; z2_NN = [y[2] for y in Y_NN]

Plots.scatter(
      z1_SN, z2_SN, 
      xlabel            = "\$z_1(x)\$",
      ylabel            = "\$z_2(x)\$",
      title             = "2-01UKP: Objective space",
      label             = "\$Y_{SN}\$",
      xlims             = (40, 80),
      ylims             = (40, 80),
      marker            = :circle,
      markersize        = 4,
      color             = :blue,
      markerstrokecolor = :blue,
      legend            = true,
      aspect_ratio      = :equal
)

Plots.scatter!(
      z1_NN, z2_NN, 
      label             = "\$Y_{NN}\$", 
      marker            = :circle,
      markersize        = 4,  
      color             = :cyan, 
      markerstrokecolor = :cyan
)

for i in 1:result_count(mo01UKP)
    y = objective_value(mo01UKP; result = i)
    Plots.annotate!(y[1] - 0.85, y[2] - 0.85, (i, 6))
end

Plots.scatter!(
      [zI[1]], [zI[2]],
      label             = "\$y^I\$",
      color             = :red, 
      markerstrokecolor = :red,
      markersize        = 5,
      marker            = :diamond
)

Plots.scatter!(
      [zN[1]], [zN[2]],
      label             = "\$y^N\$",
      color             = :orange, 
      markerstrokecolor = :orange,
      markersize        = 5,
      marker            = :diamond
)

# Rectangle corners (closed polygon: back to the start point)
rect_x = [zN[1], zI[1], zI[1], zN[1], zN[1]]
rect_y = [zN[2], zN[2], zI[2], zI[2], zN[2]]
Plots.plot!(
      rect_x, rect_y, 
      linestyle  = :dash, 
      linecolor  = :black, 
      linewidth  = 0.5, 
      label=""
)

<br>

## 8. Extending to the tri-objective case

#### • add the third objective $𝑝^3=( \ 17, \ 19,  3, \ 6, \ 12, \ 17, \ 18, \ 6, \ 6, \ 14)$ to the UKP model

In [ ]:
p3 = [17, 19, 3, 6, 12, 17, 18, 6, 6, 14]

@expression(mo01UKP, objective3, sum(p3[i] * x[i] for i in 1:n))
@objective(mo01UKP, Max, [objective1, objective2, objective3])

print(mo01UKP)

#### • solve the resulting tri-01UKP using MOA.KirlikSayin() to obtain the nondominated points $Y_N$

In [ ]:
set_attribute(mo01UKP, MOA.Algorithm(), MOA.KirlikSayin())
optimize!(mo01UKP)

In [ ]:
card_Y_N = result_count(mo01UKP)
println("|Y_N| = ", card_Y_N)
Y_N = [round.(Int, objective_value(mo01UKP; result = i)) for i in 1:card_Y_N]

#### • extract the supported nondominated points $Y_{SN}$ using `MOA.GeneralDichotomy()` (which requires the `Polyhedra` package), and derive $Y_{NN}$

In [ ]:
using Polyhedra

In [ ]:
set_attribute(mo01UKP, MOA.Algorithm(), MOA.GeneralDichotomy())
optimize!(mo01UKP)

In [ ]:
card_Y_SN = result_count(mo01UKP)
Y_SN = [round.(Int, objective_value(mo01UKP; result = i)) for i in 1:card_Y_SN]
println("|Y_SN| = ", card_Y_SN)
println("Y_SN = ", Y_SN)

Y_NN = setdiff(Y_N, Y_SN)

card_Y_NN = length(Y_NN)
println("|Y_NN| = ", card_Y_NN)
println("Y_NN = ", Y_NN)

**Comment:** The remark made above about the approach to obtaining $Y_{SN}$ also applies here. However, the algorithm is not as straightforward when $p>2$ (because there is no natural total order on $Y_N$ and no notion of a "next facet" reachable by a simple sequential sweep). The moral of the story: $2+1 \ne 3$ in multi-objective optimization.

#### • visualize $Y_{SN}$ and $Y_{NN}$ in a 3D plot of the objective space.

In [ ]:
z1_SN = [y[1] for y in Y_SN]; z2_SN = [y[2] for y in Y_SN]; z3_SN = [y[3] for y in Y_SN]
z1_NN = [y[1] for y in Y_NN]; z2_NN = [y[2] for y in Y_NN]; z3_NN = [y[3] for y in Y_NN]

Plots.scatter3d(
          z1_SN, z2_SN, z3_SN,
          xlabel            = "\$z_1(x)\$",
          ylabel            = "\$z_2(x)\$",
          zlabel            = "\$z_3(x)\$",          
          title             = "3-01UKP: Objective space",
          label             = "\$Y_{SN}\$",
          xlims             = (35, 75),
          ylims             = (45, 85),
          zlims             = (45, 85),          
          marker            = :circle,
          markersize        = 4,
          color             = :blue,
          markerstrokecolor = :blue,
          legend            = true,
          aspect_ratio      = :equal
)

Plots.scatter3d!(z1_NN, z2_NN, z3_NN,
          label             = "\$Y_{NN}\$", 
          marker            = :circle,
          markersize        = 4,  
          color             = :cyan, 
          markerstrokecolor = :cyan
)  